# HR.TRG_DEP Full Load Notebook

**Source File:** `ODI_SQL_FILE.sql`
**Conversion Timestamp:** 2024-07-30 12:00:00
**Description:** This notebook performs a full load into the `TRG_DEP` table from `DEPARTMENTS`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., L for Load)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. ETL Last Extract Time")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "6. ETL Current Extract Time")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Drop and Recreate Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30} are setup steps or no-ops in ODI, handled by full load strategy.
-- Drop existing target table for a full load.
DROP TABLE IF EXISTS workspace.hr.trg_dep;

In [ ]:
%sql
-- Recreate the target table based on the source structure.
-- Assuming data types based on common Oracle HR.DEPARTMENTS table columns:
-- DEPARTMENT_ID: NUMBER(4,0) -> BIGINT
-- DEPARTMENT_NAME: VARCHAR2(30) -> STRING
-- MANAGER_ID: NUMBER(6,0) -> BIGINT
-- LOCATION_ID: NUMBER(4,0) -> BIGINT
CREATE TABLE workspace.hr.trg_dep (
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT
) USING DELTA;

## Insert into Target Table

In [ ]:
%sql
-- Insert all records from the source DEPARTMENTS table into TRG_DEP.
-- Oracle hint /*+ APPEND PARALLEL */ is removed as it's not applicable to Delta Lake.
INSERT INTO workspace.hr.trg_dep
  (
    DEPARTMENT_ID ,
    DEPARTMENT_NAME ,
    MANAGER_ID ,
    LOCATION_ID
  )
SELECT
  DEPARTMENTS.DEPARTMENT_ID ,
  DEPARTMENTS.DEPARTMENT_NAME ,
  DEPARTMENTS.MANAGER_ID ,
  DEPARTMENTS.LOCATION_ID
FROM
  workspace.hr.departments AS DEPARTMENTS;

## Optimize Target Table

In [ ]:
%sql
-- Optimize the target table for query performance.
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_dep ZORDER BY (DEPARTMENT_ID);

## Validation

In [ ]:
%sql
SELECT
  'workspace.hr.trg_dep' AS table_name,
  COUNT(*) AS record_count
FROM
  workspace.hr.trg_dep;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 10;

## Cleanup

In [ ]:
%sql
DROP VIEW IF EXISTS v_etl_parameters;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: All schema references (`HR`) have been converted to `workspace.hr` and table names to lowercase (`trg_dep`, `departments`).
2.  **Oracle Hints**: The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Spark SQL / Delta Lake.
3.  **Full Load Strategy**: This notebook implements a full load by dropping and recreating the `TRG_DEP` table before inserting data. This aligns with the provided SQL, which only contains an `INSERT` statement without `UPDATE` or `DELETE` logic for incremental processing.
4.  **Data Types**: Explicit `CREATE TABLE` statement for `TRG_DEP` has been added. Oracle `NUMBER(p,0)` is mapped to `BIGINT` and `VARCHAR2` to `STRING`. Please verify these data types against your actual Oracle schema for `HR.DEPARTMENTS` and adjust if necessary (e.g., if a `NUMBER` column has a scale > 0, use `DECIMAL(p,s)`).
5.  **Optimization**: An `OPTIMIZE ... ZORDER BY` statement has been included to improve query performance on the `DEPARTMENT_ID` column. `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` is added for robustness (F.11).
6.  **ETL Parameters**: Placeholder `dbutils` widgets and a temporary view `v_etl_parameters` have been added to adhere to the standard notebook structure for managing ETL parameters, although they are not directly used in the provided simple `INSERT` statement.